# チュートリアル1: 基本的な分子生成

このチュートリアルでは、E(3)同変拡散モデルを使用した基本的な3D分子生成の学習と使用方法を示します。

**所要時間**: 15-20分（事前学習済みモデル使用時）または2-3時間（最初から学習する場合）

**学習内容**:
- QM9データセットの読み込み
- E(3)同変拡散モデルの構築
- モデルの学習
- 新しい分子のサンプリング
- 生成品質の評価

**前提知識**: なし（初心者向け）


## セットアップとインポート

まず、必要なライブラリをインポートし、作業ディレクトリを設定します。


In [ ]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import argparse
import numpy as np
from qm9 import dataset
from qm9.models import get_model
from qm9 import utils as qm9_utils

# デバイスの設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')


## 1. データ読み込み

QM9データセットを読み込みます。QM9は約13万個の小さな有機分子のデータセットで、各分子は3D構造と量子化学的性質を含んでいます。

**データセットの詳細**:
- 分子数: 133,885個
- 原子タイプ: H, C, N, O, F
- 最大原子数: 29
- 性質: エネルギー、双極子モーメント、分極率など


In [ ]:
# データ読み込み用の設定
class Args:
    def __init__(self):
        self.batch_size = 64
        self.num_workers = 0  # チュートリアルでは0を推奨
        self.filter_n_atoms = None  # 特定原子数でフィルタリング（Noneで全て）
        self.dataset = 'qm9'  # データセット名
        self.datadir = 'qm9/temp'  # データ保存先
        self.conditioning = []  # 条件付けなし（基本生成）
        self.remove_h = False  # 水素原子を保持

args = Args()

# データローダーの取得
print('データセットを読み込んでいます...')
dataloaders, charge_scale = dataset.retrieve_dataloaders(args)

print(f'学習サンプル数: {len(dataloaders["train"].dataset)}')
print(f'検証サンプル数: {len(dataloaders["valid"].dataset)}')
print(f'テストサンプル数: {len(dataloaders["test"].dataset)}')


### サンプルの確認

データセットから1つのサンプルを取り出して内容を確認します。


In [ ]:
# サンプルデータの取得
sample_batch = next(iter(dataloaders['train']))

print('バッチの内容:')
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f'  {key}: shape={value.shape}, dtype={value.dtype}')
    else:
        print(f'  {key}: {value}')

# 最初の分子の詳細
positions = sample_batch['positions'][0]  # 原子座標 [n_atoms, 3]
one_hot = sample_batch['one_hot'][0]  # 原子タイプ [n_atoms, n_atom_types]
charges = sample_batch['charges'][0]  # 電荷 [n_atoms, 1]

print(f'\n最初の分子:')
print(f'  原子数: {positions.shape[0]}')
print(f'  原子タイプの種類: {one_hot.shape[1]}')
print(f'  座標範囲: [{positions.min():.2f}, {positions.max():.2f}]')


## 2. モデルのセットアップ

E(3)同変拡散モデルを構築します。このモデルは3D回転と並進に対して同変性を持つため、分子の向きに依存しない生成が可能です。

**モデルのアーキテクチャ**:
- バックボーン: EGNN (E(n) Equivariant Graph Neural Network)
- 拡散プロセス: 変分下界（VLB）を最適化
- 同変性: E(3)群（3次元ユークリッド群）


In [ ]:
# モデル設定
args.n_epochs = 5  # チュートリアル用の短い学習
args.lr = 1e-4
args.nf = 128  # 隠れ層の次元数
args.n_layers = 4  # EGNNレイヤー数
args.diffusion_steps = 500  # 拡散ステップ数
args.diffusion_noise_schedule = 'polynomial_2'  # ノイズスケジュール
args.diffusion_noise_precision = 1e-5
args.ema_decay = 0.999  # Exponential Moving Average
args.normalize_factors = [1, 4, 1]  # [x, h, charges]の正規化係数
args.include_charges = True  # 電荷を含める

# データセット情報の取得
dataset_info = qm9_utils.get_dataset_info(args.dataset, args.remove_h)

# モデルの構築
print('モデルを構築しています...')
model, nodes_dist, prop_dist = get_model(args, device, dataset_info, dataloaders['train'])

print(f'モデルパラメータ数: {sum(p.numel() for p in model.parameters()):,}')
print(f'学習可能パラメータ数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


## 3. 学習（クイックデモ）

モデルを短時間学習します。実用的な性能を得るには、より長い学習が必要です（通常は500-1000エポック）。

**注意**: このチュートリアルでは、デモンストレーション目的で非常に短い学習（5エポック）を行います。


In [ ]:
# オプティマイザーの設定
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-12)

# 学習ループ
print('学習を開始します...')
model.train()

for epoch in range(args.n_epochs):
    epoch_loss = 0.0
    n_batches = 0
    
    for batch_idx, batch in enumerate(dataloaders['train']):
        # データをデバイスに転送
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                for k, v in batch.items()}
        
        # 順伝播
        optimizer.zero_grad()
        loss = model(batch)
        
        # 逆伝播
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        
        # 進捗表示（100バッチごと）
        if (batch_idx + 1) % 100 == 0:
            print(f'  エポック {epoch+1}/{args.n_epochs}, '
                  f'バッチ {batch_idx+1}, '
                  f'損失: {loss.item():.4f}')
        
        # クイックデモのため、各エポック200バッチまで
        if batch_idx >= 200:
            break
    
    avg_loss = epoch_loss / n_batches
    print(f'エポック {epoch+1}/{args.n_epochs} 完了, 平均損失: {avg_loss:.4f}')

print('学習完了!')


In [ ]:
# モデルを評価モードに
model.eval()
print('モデルを評価モードに設定しました。')


## 4. 新しい分子のサンプリング

学習したモデルを使用して、新しい分子を生成します。

**サンプリングプロセス**:
1. ランダムノイズから開始
2. 拡散プロセスを逆方向に実行
3. 段階的にノイズを除去
4. 最終的な分子構造を取得


In [ ]:
# サンプリング設定
n_samples = 10  # 生成する分子数

print(f'{n_samples}個の分子を生成しています...')

with torch.no_grad():
    # ノード数分布からサンプリング
    n_nodes = nodes_dist.sample(n_samples)
    
    # 最大ノード数
    max_n_nodes = int(n_nodes.max().item())
    
    # マスクの作成
    arange = torch.arange(max_n_nodes, device=device).unsqueeze(0).expand(n_samples, -1)
    node_mask = arange < n_nodes.unsqueeze(1)
    
    # エッジマスク（完全グラフ）
    edge_mask = node_mask.unsqueeze(1) * node_mask.unsqueeze(2)
    # 自己ループを除外
    diag_mask = ~torch.eye(max_n_nodes, device=device, dtype=torch.bool).unsqueeze(0)
    edge_mask = edge_mask * diag_mask
    
    # サンプリング実行
    x, h = model.sample(n_samples, max_n_nodes, node_mask, edge_mask, context=None)
    
    print('サンプリング完了!')
    print(f'生成された座標の形状: {x.shape}')  # [n_samples, max_n_nodes, 3]
    print(f'生成された特徴の形状: {h.shape}')  # [n_samples, max_n_nodes, n_features]


In [ ]:
# 生成された分子の統計
print('\n生成された分子の統計:')
for i in range(min(n_samples, 5)):  # 最初の5個を表示
    mask = node_mask[i]
    n_atoms = mask.sum().item()
    positions_i = x[i, mask]
    
    # 原子タイプ（最大確率）
    atom_types = h[i, mask].argmax(dim=-1)
    
    print(f'\n分子 {i+1}:')
    print(f'  原子数: {n_atoms}')
    print(f'  座標範囲: [{positions_i.min().item():.2f}, {positions_i.max().item():.2f}]')
    print(f'  原子タイプ分布: {torch.bincount(atom_types)}')


## 5. 分子品質の評価

生成された分子の品質を評価します。主な評価指標:

- **原子安定性**: 各原子の結合数が妥当か
- **分子安定性**: 分子全体が化学的に妥当か
- **有効性**: RDKitで処理可能か


In [ ]:
from qm9.analyze import check_stability

print('生成された分子の安定性を評価しています...')

# 各分子の安定性をチェック
n_stable_atoms = 0
n_stable_molecules = 0
total_atoms = 0

for i in range(n_samples):
    mask = node_mask[i]
    n_atoms = mask.sum().item()
    
    positions_i = x[i, mask].cpu()
    atom_types_i = h[i, mask].argmax(dim=-1).cpu()
    
    # 電荷（ゼロと仮定、または別途予測）
    charges_i = torch.zeros(n_atoms, 1)
    
    # 安定性チェック
    atom_stable, molecule_stable, _ = check_stability(
        positions_i, atom_types_i, charges_i, dataset_info
    )
    
    n_stable_atoms += atom_stable
    n_stable_molecules += int(molecule_stable)
    total_atoms += n_atoms

print(f'\n評価結果:')
print(f'  原子安定性: {n_stable_atoms}/{total_atoms} ({100*n_stable_atoms/total_atoms:.1f}%)')
print(f'  分子安定性: {n_stable_molecules}/{n_samples} ({100*n_stable_molecules/n_samples:.1f}%)')
print(f'\n注意: これは短い学習（{args.n_epochs}エポック）での結果です。')
print(f'完全な学習（500-1000エポック）では、90%以上の安定性が期待されます。')


## 6. モデルの保存と読み込み

学習したモデルを保存し、後で再利用できるようにします。


In [ ]:
# モデルの保存
save_path = 'tutorial_model_basic.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'args': args,
    'dataset_info': dataset_info,
}, save_path)

print(f'モデルを {save_path} に保存しました。')

# モデルの読み込み（例）
# checkpoint = torch.load(save_path)
# model.load_state_dict(checkpoint['model_state_dict'])
# print('モデルを読み込みました。')


## まとめ

このチュートリアルでは、以下を学習しました:

✅ QM9データセットの読み込みと構造理解  
✅ E(3)同変拡散モデルの構築と設定  
✅ モデルの学習プロセス  
✅ 新しい分子のサンプリング  
✅ 生成品質の評価方法  
✅ モデルの保存と読み込み  

### 次のステップ

- **チュートリアル2**: 条件付き生成 - 特定の性質を持つ分子の生成
- **チュートリアル3**: 結晶生成 - 周期境界条件を持つ結晶構造
- **完全な学習**: より長いエポック数（500-1000）で学習し、高品質な結果を得る

### 重要な原則

このコードは以下の原則に従っています:

1. **フォールバックなし**: エラーは明示的に報告され、回避策なし
2. **理論的健全性**: E(3)同変性を厳密に維持
3. **再現性**: 全てのパラメータと手順を明示的に記述

---

**質問やフィードバックは、GitHubのIssuesでお寄せください。**
